# Run RDCnet for organoid segmentation with zipping

This notebook runs a parallel processing version to apply the RDCnet for organoid segmentation on your data.

What it does:
- Convert your existing filetype (.tiff) to .zarr for low memory accesss
- Perform blockwise RDCnet segmenation of the supplied full image with certain overlap into neighbouring blocks
- Perform blockwise postprocessing
    - Remove boundaries region per segment to split touching (leaves border touching intact)
    - Per segment select only the biggest remaining connected component
    - Perform dask connected component labeling OVER blocks to relabel all segments
    - Refill new segments to a mask of the original labels removing any signal outside of this mask

In [1]:
from parallel_processing import *
from dask.distributed import Client, LocalCluster

2024-03-05 15:32:38.701638: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Provide parameters
-   Paths
-   Block information (size and overlap)
-   Parallel processing parameters

In [2]:
#####################################################
############## Input and output paths ###############
#####################################################

# Path to the image you want to segment
image_path = "/Users/samdeblank/Downloads/Emma-Dataset/WT_mix1_z2-full.zarr"

# Folder in which to store output of the segmentation
output_folder = "/Users/samdeblank/Downloads/test"

# Folder that contains the RDCnet model
rdcnet_model_folder = "/Users/samdeblank/Downloads/RCNnet_models/SMSilver"

#####################################################
################ Block parameters ##################
#####################################################

# Size of each block [z, y, x]
# Set a value to None if no blocks should be made in that dimension
block_size = [None, 512, 512]

# Overlap in pixels that each block overlaps with neighbouring blocks
block_overlap = [None, 100, 100]

#####################################################
########### Parallel processing parameters ##########
#####################################################

# Specify if you want to use GPU or CPU for processing of the image
# If you specify GPU, nr_parallel processes will be limited to 1
cpu_or_gpu = "gpu"

# How many blocks to process in parallel (Dependent on CPUs or GPUs)
nr_parallel_processes = 1

# How much memory to assign to each process (block)
# Auto automatically divides the memory over all processes
# Supply "1Gb" or similar to supply absolute value per process
process_memory_limit = "auto"

# How many cpus/threads to assign to each process
nr_cpus_per_process = 1


### Initialization
-   Initialize the Zipp3r
-   Check availability of GPUs if cpu_or_gpu is specified as "gpu"
-   Load the specified image (convert to .zarr if .tiff)
-   Output graphical overview of the iamge in blocks

In [3]:
# Initialize the zipper
block3r = Block3r(image_path, outfolder = output_folder)

# If GPU is specified, check if one or multiple GPUs are available
if cpu_or_gpu=="gpu":
    block3r.check_gpu_availablity()

# Convert supplied image to .zarr for parallel processing
# If already .zarr, loads the .zarr instead
block3r.convert_to_zarr(chunksize=block_size)

# Print the blocked overview of your segmenation
block3r.image

AssertionError: !!! No GPUs available, please check if tensorflow GPU is correctly installed and all CUDA versions match !!!

### Perform parallelized segmentation
- Initializes the dask client locally for parallelized processing
- Perform blockwise RDCnet segmentation and zip the resulting segments
- Recombine all blocks to create a single segmented image (.tiff)

In [ ]:
# Initialize the dask client to parallelize the segmentation
client = initialize_local_dask_client(
    n_workers=nr_parallel_processes,
    memory_limit=process_memory_limit,
    cpu_or_gpu=cpu_or_gpu,
    threads_per_worker=nr_cpus_per_process
    )
# Perform RDCnet segmentation using the supplied model and with provided overlap between each block
block3r.rdcnet_segmentation(
    rdcnet_model_folder=rdcnet_model_folder,
    overlap=block_overlap
)

block3r.recombine_to_tiff()
#Close the dask client
client.close()